<a href="https://colab.research.google.com/github/talhanoor23/algorithmic-trading/blob/main/Clustering_S%26P%E2%80%AF500_stocks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Load and **Data_Preprocessing**

In [ ]:
!pip install --upgrade pip
!pip uninstall -y numpy pandas_ta
!pip install numpy==1.24.3 pandas_ta

In [ ]:
from statsmodels.regression.rolling import RollingOLS
import pandas_datareader.data as web
import matplotlib.pyplot as plt
import statsmodels.api as sm
import pandas as pd
import datetime as dt
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

In [ ]:
sp500 = pd.read_html('https://en.wikipedia.org/wiki/List_of_S%26P_500_companies')[0]

In [ ]:
sp500['Symbol'] = sp500['Symbol'].str.replace('.', '-')

symbols_list = sp500['Symbol'].unique().tolist()
symbols_list

In [ ]:
len(symbols_list)

503

In [ ]:
end_date = '2023-09-27'
# start_date = pd.to_datetime(end_date) - pd.DateOffset(years=8) #same hae
start_date = pd.to_datetime(end_date) - pd.DateOffset(365*8)
start_date

Timestamp('2015-09-29 00:00:00')

In [ ]:
import yfinance as yf
import pandas as pd

# df = yf.download(tickers=symbols_list, start=start_date, end=end_date, group_by='ticker', auto_adjust=False)
df = yf.download(tickers=symbols_list, start=start_date, end=end_date, auto_adjust=False)

[*********************100%***********************]  503 of 503 completed
ERROR:yfinance:
4 Failed downloads:
ERROR:yfinance:['SW', 'SOLV', 'GEV', 'VLTO']: YFPricesMissingError('possibly delisted; no price data found  (1d 2015-09-29 00:00:00 -> 2023-09-27) (Yahoo error = "Data doesn\'t exist for startDate = 1443499200, endDate = 1695787200")')


In [ ]:
print(df.columns.levels)

[['Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume'], ['A', 'AAPL', 'ABBV', 'ABNB', 'ABT', 'ACGL', 'ACN', 'ADBE', 'ADI', 'ADM', 'ADP', 'ADSK', 'AEE', 'AEP', 'AES', 'AFL', 'AIG', 'AIZ', 'AJG', 'AKAM', 'ALB', 'ALGN', 'ALL', 'ALLE', 'AMAT', 'AMCR', 'AMD', 'AME', 'AMGN', 'AMP', 'AMT', 'AMZN', 'ANET', 'ANSS', 'AON', 'AOS', 'APA', 'APD', 'APH', 'APO', 'APTV', 'ARE', 'ATO', 'AVB', 'AVGO', 'AVY', 'AWK', 'AXON', 'AXP', 'AZO', 'BA', 'BAC', 'BALL', 'BAX', 'BBY', 'BDX', 'BEN', 'BF-B', 'BG', 'BIIB', 'BK', 'BKNG', 'BKR', 'BLDR', 'BLK', 'BMY', 'BR', 'BRK-B', 'BRO', 'BSX', 'BX', 'BXP', 'C', 'CAG', 'CAH', 'CARR', 'CAT', 'CB', 'CBOE', 'CBRE', 'CCI', 'CCL', 'CDNS', 'CDW', 'CEG', 'CF', 'CFG', 'CHD', 'CHRW', 'CHTR', 'CI', 'CINF', 'CL', 'CLX', 'CMCSA', 'CME', 'CMG', 'CMI', 'CMS', 'CNC', ...]]


In [ ]:
df

Price        Adj Close                                                 \
Ticker               A        AAPL        ABBV        ABNB        ABT   
Date                                                                    
2015-09-29   31.251011   24.536383   35.061218         NaN  32.820755   
2015-09-30   31.797489   24.815361   36.137154         NaN  33.419006   
2015-10-01   31.251011   24.653374   36.615360         NaN  33.867695   
2015-10-02   32.112408   24.833363   37.073643         NaN  34.382851   
2015-10-05   32.732983   24.923355   37.644817         NaN  34.391155   
...                ...         ...         ...         ...        ...   
2023-09-20  111.046104  173.985001  144.022476  138.009995  96.817795   
2023-09-21  108.340805  172.438400  143.853714  132.750000  95.514397   
2023-09-22  109.999535  173.290970  143.206772  132.199997  94.761330   
2023-09-25  110.522812  174.569916  144.997559  134.139999  94.104820   
2023-09-26  109.328133  170.485260  144.303757  132.279999  92.907623   

Price                                                                 ...  \
Ticker           ACGL         ACN        ADBE         ADI        ADM  ...   
Date                                                                  ...   
2015-09-29  23.217773   82.010071   79.830002   44.396568  30.731060  ...   
2015-09-30  23.287504   83.419380   82.220001   46.335064  31.053204  ...   
2015-10-01  23.262148   83.572174   83.750000   45.127609  31.060698  ...   
2015-10-02  23.563267   84.378693   83.989998   45.743664  31.779907  ...   
2015-10-05  23.766125   86.271889   85.839996   47.172886  32.693897  ...   
...               ...         ...         ...         ...        ...  ...   
2023-09-20  77.327065  307.660461  535.780029  170.806671  74.535332  ...   
2023-09-21  76.347641  302.998962  513.880005  168.633759  74.065849  ...   
2023-09-22  76.984741  306.970978  512.900024  170.224625  72.281822  ...   
2023-09-25  77.165421  307.844971  511.600006  170.176132  72.582283  ...   
2023-09-26  77.270020  301.649048  506.299988  167.178665  71.427361  ...   

Price       Volume                                                          \
Ticker         WTW       WY      WYNN      XEL       XOM      XYL      YUM   
Date                                                                         
2015-09-29  345752  3053000   6223400  3361700  13754400  1344100  6018857   
2015-09-30  649262  2837500   4265300  2722000  18176500  1708500  7378142   
2015-10-01  527632  3270800   4745600  2875200  14397800  1498600  4426718   
2015-10-02  367458  2846400  15153700  3299500  18811300  1451100  7719772   
2015-10-05  179275  3286600  12699700  5369400  18486700  2376800  5737736   
...            ...      ...       ...      ...       ...      ...      ...   
2023-09-20  670700  2374200   1061300  7670600  11879900  1752100  1437200   
2023-09-21  620600  3494500   2358400  4904800  14400600  2044800  1492700   
2023-09-22  651100  3476300   2611300  4904300  13197300  1699800  1534100   
2023-09-25  477500  2269600   1387800  3617800  11316000  1702000  1048500   
2023-09-26  993600  3324700   1454000  6275500  11805400  1322400  1500600   

Price                                 
Ticker          ZBH    ZBRA      ZTS  
Date                                  
2015-09-29  2358906  895000  7328400  
2015-09-30  2012723  618800  4954100  
2015-10-01  1604019  464900  4131800  
2015-10-02  1353626  341500  4036800  
2015-10-05  1352081  460600  5467700  
...             ...     ...      ...  
2023-09-20  1055900  824900  1485500  
2023-09-21  1489200  676200  1691800  
2023-09-22  1267300  482200  1281800  
2023-09-25   886700  314500  1551100  
2023-09-26  3610500  355400  1463200  

[2012 rows x 3018 columns]

In [ ]:
# df_n = df.stack(level=0).reset_index()
df_n = df.stack().reset_index()
df_n
#best but didn't match to professors dataframe.....

Price,Date,Ticker,Adj Close,Close,High,Low,Open,Volume
0,2015-09-29,A,31.251011,33.740002,34.060001,33.240002,33.360001,2252400.0
1,2015-09-29,AAPL,24.536383,27.264999,28.377501,26.965000,28.207500,293461600.0
2,2015-09-29,ABBV,35.061218,52.790001,54.189999,51.880001,53.099998,12842800.0
3,2015-09-29,ABT,32.820755,39.500000,40.150002,39.029999,39.259998,12287500.0
4,2015-09-29,ACGL,23.217773,24.416668,24.456667,24.100000,24.170000,1888800.0
...,...,...,...,...,...,...,...,...
980413,2023-09-26,XYL,87.701073,89.519997,90.849998,89.500000,90.379997,1322400.0
980414,2023-09-26,YUM,119.860718,124.010002,124.739998,123.449997,124.239998,1500600.0
980415,2023-09-26,ZBH,110.800163,112.459999,117.110001,112.419998,116.769997,3610500.0
980416,2023-09-26,ZBRA,223.960007,223.960007,226.649994,222.580002,225.970001,355400.0


In [ ]:
df_nn = df.stack()
df_nn

Price               Adj Close       Close        High         Low        Open  \
Date       Ticker                                                               
2015-09-29 A        31.251011   33.740002   34.060001   33.240002   33.360001   
           AAPL     24.536383   27.264999   28.377501   26.965000   28.207500   
           ABBV     35.061218   52.790001   54.189999   51.880001   53.099998   
           ABT      32.820755   39.500000   40.150002   39.029999   39.259998   
           ACGL     23.217773   24.416668   24.456667   24.100000   24.170000   
...                       ...         ...         ...         ...         ...   
2023-09-26 XYL      87.701073   89.519997   90.849998   89.500000   90.379997   
           YUM     119.860718  124.010002  124.739998  123.449997  124.239998   
           ZBH     110.800163  112.459999  117.110001  112.419998  116.769997   
           ZBRA    223.960007  223.960007  226.649994  222.580002  225.970001   
           ZTS     173.604691  176.869995  178.449997  176.270004  176.580002   

Price                   Volume  
Date       Ticker               
2015-09-29 A         2252400.0  
           AAPL    293461600.0  
           ABBV     12842800.0  
           ABT      12287500.0  
           ACGL      1888800.0  
...                        ...  
2023-09-26 XYL       1322400.0  
           YUM       1500600.0  
           ZBH       3610500.0  
           ZBRA       355400.0  
           ZTS       1463200.0  

[980418 rows x 6 columns]

In [ ]:
df_nn.columns.name = None #for dropping price tag....
df_nn.index.names = ['date', 'tickers']
df_nn.columns = df_nn.columns.str.lower()
df_nn
#exactly like  the professor tough in youtube.........

adj close       close        high         low  \
date       tickers                                                   
2015-09-29 A         31.251011   33.740002   34.060001   33.240002   
           AAPL      24.536383   27.264999   28.377501   26.965000   
           ABBV      35.061218   52.790001   54.189999   51.880001   
           ABT       32.820755   39.500000   40.150002   39.029999   
           ACGL      23.217773   24.416668   24.456667   24.100000   
...                        ...         ...         ...         ...   
2023-09-26 XYL       87.701073   89.519997   90.849998   89.500000   
           YUM      119.860718  124.010002  124.739998  123.449997   
           ZBH      110.800163  112.459999  117.110001  112.419998   
           ZBRA     223.960007  223.960007  226.649994  222.580002   
           ZTS      173.604691  176.869995  178.449997  176.270004   

                          open       volume  
date       tickers                           
2015-09-29 A         33.360001    2252400.0  
           AAPL      28.207500  293461600.0  
           ABBV      53.099998   12842800.0  
           ABT       39.259998   12287500.0  
           ACGL      24.170000    1888800.0  
...                        ...          ...  
2023-09-26 XYL       90.379997    1322400.0  
           YUM      124.239998    1500600.0  
           ZBH      116.769997    3610500.0  
           ZBRA     225.970001     355400.0  
           ZTS      176.580002    1463200.0  

[980418 rows x 6 columns]

# **Calculate features and technical indicators for each stock.**